# 🔍 Chandra OCR — Invoice & Table Extraction Demo

---

> **Purpose:** Upload one or more images (invoices, tables, forms, receipts, etc.),  
> extract structured data using the **Datalab Chandra OCR API**,  
> and download all results as a formatted **Excel workbook**.

---

## 📋 What this notebook does

| Step | Description |
|------|-------------|
| **1. Setup** | Install dependencies & configure API key from Colab Secrets |
| **2. Upload** | Upload single or multiple image files (JPG, PNG, TIFF, etc.) |
| **3. Extract** | Two modes: **Convert** (full markdown+JSON) or **Structured Extract** (schema-based) |
| **4. Review** | Preview extracted tables in the notebook |
| **5. Export** | Download a formatted `.xlsx` file with all results |

---

## 🔑 Before you start — add your API key

1. In the left sidebar, click the **🔑 key icon** ("Secrets")
2. Click **"+ Add new secret"**
3. Set **Name** = `CHANDRA_API_KEY`
4. Paste your Datalab API key as the **Value**
5. Toggle **"Notebook access"** ON

> Get your API key at: **https://www.datalab.to/app/keys**

---

## 🌐 API Endpoints Used

| Endpoint | Purpose |
|----------|---------|
| `POST https://www.datalab.to/api/v1/convert` | Converts image → Markdown + JSON (tables, text, layout) |
| `POST https://www.datalab.to/api/v1/extract` | Schema-based structured extraction (invoice fields, line items) |
| `GET  https://www.datalab.to/api/v1/{endpoint}/{request_id}` | Poll for results (async pattern) |

---
## ⚙️ Step 1 — Install Dependencies & Load API Key

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Install required packages
#   openpyxl  → Excel file creation and formatting
#   Pillow    → Image validation and conversion
#   requests  → HTTP calls to Datalab API
#   pandas    → Data manipulation and preview
# ─────────────────────────────────────────────────────────────────────────────
!pip install openpyxl Pillow requests pandas --quiet

print("✅ Dependencies installed successfully.")

✅ Dependencies installed successfully.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Load the Datalab API key securely from Colab Secrets
#
# HOW TO ADD YOUR KEY:
#   Left sidebar → 🔑 Secrets → + Add new secret
#   Name:  CHANDRA_API_KEY
#   Value: <your key from https://www.datalab.to/app/keys>
#   Toggle "Notebook access" ON
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import userdata

try:
    CHANDRA_API_KEY = userdata.get('CHANDRA_API_KEY')
    if not CHANDRA_API_KEY:
        raise ValueError("Secret exists but is empty.")
    print(f"✅ API key loaded.  (ends with: ...{CHANDRA_API_KEY[-4:]})")
except Exception as e:
    CHANDRA_API_KEY = None
    print(f"❌ Could not load CHANDRA_API_KEY: {e}")
    print("   Please follow the setup instructions in the header cell above.")

✅ API key loaded.  (ends with: ...vLYo)


---
## 📸 Step 2 — Upload Your Images

Run the cell below, then use the **"Choose Files"** button to upload one or more images.

**Supported formats:** JPG, JPEG, PNG, TIFF, BMP, WEBP  
**Max file size:** 10 MB per image  
**Tip:** You can select multiple files at once with Ctrl+Click (or Cmd+Click on Mac)

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Upload images via Colab's built-in file picker
# ─────────────────────────────────────────────────────────────────────────────
from google.colab import files
from pathlib import Path
from IPython.display import display, HTML

print("📂 Please select one or more image files to upload...")
uploaded = files.upload()

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tiff", ".tif", ".bmp", ".webp"}
MAX_SIZE_MB = 10

valid_images = []
skipped      = []

for filename, file_bytes in uploaded.items():
    ext     = Path(filename).suffix.lower()
    size_mb = len(file_bytes) / (1024 * 1024)
    if ext not in SUPPORTED_EXTENSIONS:
        skipped.append((filename, f"Unsupported format '{ext}'"))
    elif size_mb > MAX_SIZE_MB:
        skipped.append((filename, f"File too large ({size_mb:.1f} MB)"))
    else:
        valid_images.append((filename, file_bytes))

print(f"\n{'─'*55}")
print(f"✅ Valid images ready for processing: {len(valid_images)}")
for name, data in valid_images:
    print(f"   📄 {name}  ({len(data)/(1024*1024):.2f} MB)")

if skipped:
    print(f"\n⚠️  Skipped ({len(skipped)}):")
    for name, reason in skipped:
        print(f"   ⛔ {name} — {reason}")

if not valid_images:
    print("\n❌ No valid images found. Re-run this cell and upload supported image files.")

📂 Please select one or more image files to upload...


Saving G2.jpeg to G2.jpeg

───────────────────────────────────────────────────────
✅ Valid images ready for processing: 1
   📄 G2.jpeg  (0.26 MB)


---
## 🤖 Step 3 — Run Chandra OCR Extraction

### Choose your extraction mode:

| Mode | Best for | Output |
|------|----------|--------|
| **`convert`** | Any document — gets all text, tables, layout | Markdown + structured JSON blocks |
| **`extract`** | Invoices, receipts, forms — pulls specific fields | Named fields from your schema |

Both modes run by default. You can disable either by setting `RUN_CONVERT = False` or `RUN_EXTRACT = False`.

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# ✏️  CONFIGURATION — Edit these settings before running extraction
# ─────────────────────────────────────────────────────────────────────────────

# ── Mode toggles ──────────────────────────────────────────────────────────────
RUN_CONVERT = True   # Convert mode: full text + all tables as JSON
RUN_EXTRACT = True   # Extract mode: schema-based structured fields

# ── Convert mode settings ─────────────────────────────────────────────────────
# Mode: 'fast' | 'balanced' | 'accurate'
# 'accurate' gives best results for complex invoices; 'fast' is quickest
CONVERT_MODE = "accurate"

# Output format for convert: 'json' gives structured blocks; 'markdown' gives readable text
CONVERT_OUTPUT_FORMAT = "json"   # 'json' recommended for Excel export

# ── Structured Extract schema ─────────────────────────────────────────────────
# This schema tells the API which specific fields to pull out.
# Customise it for your document type — add/remove fields as needed.
# Tip: The more descriptive your field descriptions, the more accurate extraction is.
# EXTRACT_SCHEMA = {
#     "vendor_name":     {"type": "string",  "description": "Name of the seller or supplier"},
#     "invoice_number":  {"type": "string",  "description": "Invoice or bill number"},
#     "invoice_date":    {"type": "string",  "description": "Date of the invoice"},
#     "gstin":           {"type": "string",  "description": "GST Identification Number of the seller"},
#     "dl_number":       {"type": "string",  "description": "Drug License number"},
#     "grand_total":     {"type": "number",  "description": "Final grand total payable amount"},
#     "total_qty":       {"type": "number",  "description": "Total quantity of all items"},
#     "sgst_payable":    {"type": "number",  "description": "SGST payable amount"},
#     "cgst_payable":    {"type": "number",  "description": "CGST payable amount"},
#     "discount":        {"type": "number",  "description": "Total discount applied"},
#     "line_items": {
#         "type": "array",
#         "description": "List of all product/medicine line items in the invoice",
#         "items": {
#             "type": "object",
#             "properties": {
#                 "sn":          {"type": "string",  "description": "Serial number"},
#                 "mfr":         {"type": "string",  "description": "Manufacturer name"},
#                 "product":     {"type": "string",  "description": "Product or medicine name"},
#                 "pack":        {"type": "string",  "description": "Pack size (e.g. 10ML, 30ML)"},
#                 "hsn":         {"type": "string",  "description": "HSN code"},
#                 "batch":       {"type": "string",  "description": "Batch number"},
#                 "qty":         {"type": "number",  "description": "Quantity ordered"},
#                 "free":        {"type": "number",  "description": "Free quantity"},
#                 "expiry":      {"type": "string",  "description": "Expiry date"},
#                 "mrp":         {"type": "number",  "description": "Maximum Retail Price"},
#                 "rate":        {"type": "number",  "description": "Rate per unit"},
#                 "dis":         {"type": "number",  "description": "Discount percentage"},
#                 "sgst":        {"type": "number",  "description": "SGST percentage"},
#                 "cgst":        {"type": "number",  "description": "CGST percentage"},
#                 "amount":      {"type": "number",  "description": "Line item total amount"}
#             }
#         }
#     }
# }
EXTRACT_SCHEMA = {
    "type": "object",
    "properties": {
        "items": {
            "type": "array",
            "description": "List of GRN medicine line items",
            "items": {
                "type": "object",
                "properties": {
                    "medicine_name": {
                        "type": "string",
                        "description": "Medicine name"
                    },
                    "batch_no": {
                        "type": ["string", "null"],
                        "description": "Batch number"
                    },
                    "expiry_date": {
                        "type": ["string", "null"],
                        "description": "Expiry date"
                    },
                    "quantity": {
                        "type": "number",
                        "description": "Quantity"
                    },
                    "free_quantity": {
                        "type": "number",
                        "description": "Free quantity"
                    },
                    "rate": {
                        "type": "number",
                        "description": "Unit rate"
                    },
                    "mrp": {
                        "type": "number",
                        "description": "MRP"
                    },
                    "gst_percent": {
                        "type": "number",
                        "description": "GST percent"
                    },
                    "amount": {
                        "type": "number",
                        "description": "Line amount"
                    },
                    "hsn_code": {
                        "type": ["string", "null"],
                        "description": "HSN code"
                    }
                },
                "required": [
                    "medicine_name"
                ]
            }
        }
    },
    "required": ["items"]
}
# ── Polling settings ──────────────────────────────────────────────────────────
POLL_INTERVAL_SEC = 3    # Seconds between status checks
TIMEOUT_SEC       = 180  # Max wait per image (seconds)

print("✅ Configuration ready.")
print(f"   Convert mode  : {'ON — format=' + CONVERT_OUTPUT_FORMAT + ', mode=' + CONVERT_MODE if RUN_CONVERT else 'OFF'}")
print(f"   Extract mode  : {'ON — schema has ' + str(len(EXTRACT_SCHEMA)) + ' top-level fields' if RUN_EXTRACT else 'OFF'}")

✅ Configuration ready.
   Convert mode  : ON — format=json, mode=accurate
   Extract mode  : ON — schema has 3 top-level fields


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Datalab API client
#
# Implements the standard Datalab async pattern:
#   1. POST to submit  → get request_check_url
#   2. GET poll loop   → wait for status == 'complete'
#   3. Parse response  → normalise into app-friendly dict
#
# Two endpoints:
#   /api/v1/convert  — full document conversion (text + tables + images)
#   /api/v1/extract  — schema-driven structured field extraction
# ─────────────────────────────────────────────────────────────────────────────

import io, json, time, re
from PIL import Image
import requests
from typing import Dict, Any, Optional

BASE_URL = "https://www.datalab.to/api/v1"


class DatalabAPIError(Exception):
    pass


def _to_jpeg(raw_bytes: bytes, filename: str) -> bytes:
    """Normalise any PIL-supported image to JPEG for the API."""
    try:
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=95)
        return buf.getvalue()
    except Exception as e:
        raise DatalabAPIError(f"Cannot decode image '{filename}': {e}")


def _poll(check_url: str, headers: dict, timeout: int, interval: float) -> Dict[str, Any]:
    """Poll check_url until status is 'complete' or timeout is exceeded."""
    deadline = time.time() + timeout
    dots = 0
    while time.time() < deadline:
        try:
            resp = requests.get(check_url, headers=headers, timeout=15)
            if resp.status_code == 429:
                raise DatalabAPIError("Rate limit exceeded (429). Please wait and retry.")
            if resp.status_code != 200:
                raise DatalabAPIError(f"Poll error [{resp.status_code}]: {resp.text[:200]}")
            data   = resp.json()
            status = data.get("status", "")
            if status == "complete":
                print(" done ✅")
                return data
            elif status == "failed":
                print(" failed ❌")
                raise DatalabAPIError(f"Server extraction failed: {data.get('error', 'unknown')}")
        except requests.exceptions.RequestException as e:
            pass  # transient network error, keep retrying
        print(".", end="", flush=True)
        dots += 1
        time.sleep(interval)
    raise DatalabAPIError(f"Timeout after {timeout}s waiting for OCR result.")


def call_convert(img_bytes: bytes, filename: str, api_key: str,
                 output_format: str = "json", mode: str = "accurate",
                 timeout: int = TIMEOUT_SEC, interval: float = POLL_INTERVAL_SEC) -> Dict[str, Any]:
    """
    POST to /api/v1/convert — converts image to markdown/html/json.

    Returns the completed poll response dict.
    Key response fields:
      result['json']      → structured blocks (when output_format='json')
      result['markdown']  → full markdown text
      result['page_count'], result['parse_quality_score']
    """
    headers = {"X-API-Key": api_key}
    jpeg    = _to_jpeg(img_bytes, filename)

    print(f"   [convert] Submitting... ", end="", flush=True)
    resp = requests.post(
        f"{BASE_URL}/convert",
        headers=headers,
        files={"file": ("image.jpg", jpeg, "image/jpeg")},
        data={"output_format": output_format, "mode": mode},
        timeout=30
    )
    if resp.status_code == 429:
        raise DatalabAPIError("Rate limit exceeded on submit.")
    if resp.status_code != 200:
        raise DatalabAPIError(f"Submit failed [{resp.status_code}]: {resp.text[:300]}")

    check_url = resp.json().get("request_check_url")
    if not check_url:
        raise DatalabAPIError("No request_check_url in submit response.")

    print("polling ", end="", flush=True)
    return _poll(check_url, headers, timeout, interval)


def call_extract(img_bytes: bytes, filename: str, api_key: str,
                 schema: dict, mode: str = "accurate",
                 timeout: int = TIMEOUT_SEC, interval: float = POLL_INTERVAL_SEC) -> Dict[str, Any]:
    """
    POST to /api/v1/extract — schema-based structured field extraction.

    Returns the completed poll response dict.
    Key response field:
      result['extraction_schema_json'] → dict of extracted fields matching your schema
    """
    headers = {"X-API-Key": api_key}
    jpeg    = _to_jpeg(img_bytes, filename)

    print(f"   [extract] Submitting... ", end="", flush=True)
    resp = requests.post(
        f"{BASE_URL}/extract",
        headers=headers,
        files={"file": ("image.jpg", jpeg, "image/jpeg")},
        data={"page_schema": json.dumps(schema), "mode": mode},
        timeout=30
    )
    if resp.status_code == 429:
        raise DatalabAPIError("Rate limit exceeded on submit.")
    if resp.status_code != 200:
        raise DatalabAPIError(f"Submit failed [{resp.status_code}]: {resp.text[:300]}")

    check_url = resp.json().get("request_check_url")
    if not check_url:
        raise DatalabAPIError("No request_check_url in extract submit response.")

    print("polling ", end="", flush=True)
    return _poll(check_url, headers, timeout, interval)


print("✅ API client functions defined.")

✅ API client functions defined.


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Process all uploaded images
#
# Results stored in `all_results` — list of dicts, one per image:
#   {
#     'filename':       str,
#     'convert_result': dict | None,   # from /api/v1/convert
#     'extract_result': dict | None,   # from /api/v1/extract
#     'errors':         list[str]
#   }
# ─────────────────────────────────────────────────────────────────────────────

if not CHANDRA_API_KEY:
    print("❌ API key not loaded. Please complete Step 1 first.")
elif not valid_images:
    print("❌ No valid images. Please complete Step 2 first.")
else:
    all_results = []
    total       = len(valid_images)
    print(f"🚀 Processing {total} image(s) with Chandra OCR...\n")

    for idx, (filename, file_bytes) in enumerate(valid_images, start=1):
        print(f"[{idx}/{total}] ─── {filename}")
        t_start  = time.time()
        record   = {"filename": filename, "convert_result": None, "extract_result": None, "errors": []}

        # ── Convert endpoint ──────────────────────────────────────────────────
        if RUN_CONVERT:
            try:
                record["convert_result"] = call_convert(
                    file_bytes, filename, CHANDRA_API_KEY,
                    output_format=CONVERT_OUTPUT_FORMAT, mode=CONVERT_MODE
                )
                score = record["convert_result"].get("parse_quality_score", "—")
                print(f"        Quality score: {score}/5")
            except Exception as e:
                record["errors"].append(f"convert: {e}")
                print(f"        ⚠️  Convert error: {e}")

        # ── Extract endpoint ──────────────────────────────────────────────────
        if RUN_EXTRACT:
            try:
                record["extract_result"] = call_extract(
                    file_bytes, filename, CHANDRA_API_KEY,
                    schema=EXTRACT_SCHEMA, mode=CONVERT_MODE
                )
                extracted = record["extract_result"].get("extraction_schema_json") or {}
                if isinstance(extracted, str):
                    extracted = json.loads(extracted)
                line_items = extracted.get("line_items", [])
                print(f"        Line items extracted: {len(line_items)}")
            except Exception as e:
                record["errors"].append(f"extract: {e}")
                print(f"        ⚠️  Extract error: {e}")

        elapsed = time.time() - t_start
        all_results.append(record)
        print(f"        ⏱️  Total time: {elapsed:.1f}s\n")

    ok    = sum(1 for r in all_results if not r["errors"])
    erred = len(all_results) - ok
    print("─" * 55)
    print(f"✅ Done.  {ok} succeeded, {erred} with errors.")

🚀 Processing 1 image(s) with Chandra OCR...

[1/1] ─── G2.jpeg
   [convert] Submitting... polling  done ✅
        Quality score: None/5
   [extract] Submitting... polling .............. done ✅
        Line items extracted: 0
        ⏱️  Total time: 54.0s

───────────────────────────────────────────────────────
✅ Done.  1 succeeded, 0 with errors.


---
## 🔎 Step 4 — Preview Extracted Data

Review the extracted fields and line items for each image before exporting.

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Helper: parse extracted schema JSON from the API response
# ─────────────────────────────────────────────────────────────────────────────
import json
import re
import pandas as pd
from IPython.display import display, HTML


def get_extracted_fields(result: dict) -> dict:
    """
    Safely normalize Chandra extract response.

    Supports:
    - direct {"items": [...]}
    - extraction_schema_json as dict
    - extraction_schema_json as JSON string
    """

    # NEW FORMAT → direct items already present
    if "items" in result:
        return result

    raw = result.get("extraction_schema_json", {})

    if isinstance(raw, str):
        raw = raw.strip()

        # Remove markdown fences if present
        raw = re.sub(r"^```[a-zA-Z]*\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)

        try:
            return json.loads(raw)
        except Exception:
            return {}

    return raw or {}


def clean_line_item(item: dict) -> dict:
    """
    Remove Chandra metadata fields like:
    *_citations
    *_score
    """

    allowed_fields = [
        "medicine_name",
        "batch_no",
        "expiry_date",
        "quantity",
        "free_quantity",
        "rate",
        "mrp",
        "gst_percent",
        "amount",
        "hsn_code",
    ]

    return {
        field: item.get(field)
        for field in allowed_fields
    }


def display_styled(df: pd.DataFrame, title: str = ""):
    """Display DataFrame with styling."""

    if title:
        display(
            HTML(
                f"<h4 style='margin:12px 0 4px; color:#2d3e50'>{title}</h4>"
            )
        )

    if df.empty:
        display(HTML("<i style='color:#888'>No data.</i>"))
        return

    styled = (
        df.style
        .set_table_styles([
            {
                "selector": "thead th",
                "props": [
                    ("background-color", "#2d3e50"),
                    ("color", "white"),
                    ("font-weight", "bold"),
                    ("padding", "6px 10px"),
                ],
            },
            {
                "selector": "tbody td",
                "props": [
                    ("padding", "5px 10px"),
                    ("border", "1px solid #ddd"),
                ],
            },
            {
                "selector": "tbody tr:nth-child(even)",
                "props": [
                    ("background-color", "#ebf5fb"),
                ],
            },
        ])
        .hide(axis="index")
    )

    display(styled)


# ─────────────────────────────────────────────────────────────────────────────
# Preview all results
# ─────────────────────────────────────────────────────────────────────────────
if 'all_results' not in dir() or not all_results:
    print("⚠️  No results yet. Please run Step 3 first.")

else:
    for record in all_results:

        fname = record["filename"]

        print(f"\n{'═'*65}")
        print(f"  📄  {fname}")
        print(f"{'═'*65}")

        # ─────────────────────────────────────────────────────────────────────
        # Errors
        # ─────────────────────────────────────────────────────────────────────
        if record.get("errors"):
            for err in record["errors"]:
                print(f"  ❌ Error: {err}")

        # ─────────────────────────────────────────────────────────────────────
        # Convert Result
        # ─────────────────────────────────────────────────────────────────────
        conv = record.get("convert_result")

        if conv:

            score = conv.get("parse_quality_score", "—")
            pages = conv.get("page_count", 1)

            print(
                f"\n  📊 Convert result — quality: {score}/5, "
                f"pages: {pages}"
            )

            md = conv.get("markdown", "")

            if md:
                print("\n  📝 Markdown preview:")
                print("  " + md[:500].replace("\n", "\n  "))

        # ─────────────────────────────────────────────────────────────────────
        # Extract Result
        # ─────────────────────────────────────────────────────────────────────
        ext = record.get("extract_result")

        if ext:

            fields = get_extracted_fields(ext)

            # Chandra returns items directly
            raw_items = fields.get("items", [])

            # Clean metadata fields
            line_items = [
                clean_line_item(item)
                for item in raw_items
            ]

            print(f"\n  🧾 Line items extracted: {len(line_items)}")

            # Display line items
            if line_items:

                items_df = pd.DataFrame(line_items)

                display_styled(
                    items_df,
                    f"🧾 GRN Line Items ({len(line_items)} rows)"
                )

            else:
                print("  ⚠️  No line items found in extracted data.")

            # Store cleaned rows for export
            record["line_items"] = line_items


═════════════════════════════════════════════════════════════════
  📄  G2.jpeg
═════════════════════════════════════════════════════════════════

  📊 Convert result — quality: None/5, pages: 1

  🧾 Line items extracted: 10


medicine_name,batch_no,expiry_date,quantity,free_quantity,rate,mrp,gst_percent,amount,hsn_code
AMCLAZEN DROPS,XD379F,7/26,5,2,99.290000,139.000000,6.000000,496.450000,30041030
AMCLAZEN-DS 457,ACAH24044D,7/26,5,2,127.860000,179.000000,6.000000,639.300000,30049099
CEZENIX 100MG DRY SYP,XD593,1/27,5,2,57.360000,80.300000,6.000000,286.800000,30041030
CEZENIX-DROPS,XD250A,1/26,5,2,99.290000,139.000000,6.000000,496.450000,30042019
CEZODOX DT 100 TAB,NPD-6780,1/26,4,0,89.290000,125.000000,6.000000,357.160000,30042019
CEZODOX-100 DRY SYP,NPD.25040,5/27,10,5,117.860000,165.000000,6.000000,1178.600000,30041030
CUREPAR-100,NCP25048,6/27,5,2,27.360000,38.300000,6.000000,136.800000,30049066
KALTCLEAR NASAL DROPS,N5F022,5/27,5,2,50.710000,71.000000,6.000000,253.550000,30049099
KALTCLEAR CP DROPS,DSL0252,5/26,10,5,67.860000,95.000000,6.000000,678.600000,300410
NAPPIX OINT 20GM,2506153,5/27,5,1,89.290000,125.000000,6.000000,446.450000,30049099


---
## 📥 Step 5 — Export to Excel & Download

All extracted data is compiled into a formatted Excel workbook with these sheets:

| Sheet | Contents |
|-------|----------|
| **Summary** | One row per image — quality score, line item count, status |
| **All Line Items** | Every extracted line item from every image combined |
| **`<filename>` — Header** | Invoice header fields for each image |
| **`<filename>` — Items** | Line items table for each image |

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# Build and download formatted Excel workbook
# ─────────────────────────────────────────────────────────────────────────────

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime
from pathlib import Path
from google.colab import files as colab_files


# ─────────────────────────────────────────────────────────────────────────────
# Styles
# ─────────────────────────────────────────────────────────────────────────────

HDR_FILL    = PatternFill("solid", fgColor="2D3E50")
ALT_FILL    = PatternFill("solid", fgColor="EBF5FB")
WHT_FILL    = PatternFill("solid", fgColor="FFFFFF")
OK_FILL     = PatternFill("solid", fgColor="D5F5E3")
ERR_FILL    = PatternFill("solid", fgColor="FADBD8")
TITLE_FILL  = PatternFill("solid", fgColor="EBF5FB")

HDR_FONT    = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT   = Font(name="Arial", size=10)
TITLE_FONT  = Font(name="Arial", bold=True, size=12, color="2D3E50")

THIN        = Side(style="thin", color="CCCCCC")
BORDER      = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

CTR         = Alignment(horizontal="center", vertical="center")
LFT         = Alignment(horizontal="left", vertical="center")


# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _cell(ws, row, col, value="", font=None, fill=None,
          align=None, border=True):

    c = ws.cell(row=row, column=col, value=value)

    if font:
        c.font = font

    if fill:
        c.fill = fill

    if align:
        c.alignment = align

    if border:
        c.border = BORDER

    return c


def write_table(ws, headers, rows, start_row=1):

    for ci, h in enumerate(headers, 1):
        _cell(
            ws,
            start_row,
            ci,
            h,
            font=HDR_FONT,
            fill=HDR_FILL,
            align=CTR
        )

    for ri, row_data in enumerate(rows, 1):

        fill = ALT_FILL if ri % 2 == 0 else WHT_FILL

        for ci, val in enumerate(row_data, 1):

            _cell(
                ws,
                start_row + ri,
                ci,
                val,
                font=BODY_FONT,
                fill=fill,
                align=LFT
            )

    # Auto width
    for ci, h in enumerate(headers, 1):

        values = [str(h)] + [
            str(r[ci - 1]) if ci - 1 < len(r) else ""
            for r in rows
        ]

        width = min(
            max(len(v) for v in values) + 3,
            40
        )

        ws.column_dimensions[
            get_column_letter(ci)
        ].width = max(width, 10)

    return start_row + len(rows)


def title_block(ws, text, subtitle="", merge_cols=10):

    ws.merge_cells(
        start_row=1,
        start_column=1,
        end_row=1,
        end_column=merge_cols
    )

    c = ws.cell(row=1, column=1, value=text)

    c.font = TITLE_FONT
    c.fill = TITLE_FILL
    c.alignment = LFT

    if subtitle:

        ws.merge_cells(
            start_row=2,
            start_column=1,
            end_row=2,
            end_column=merge_cols
        )

        c2 = ws.cell(row=2, column=1, value=subtitle)

        c2.font = Font(
            name="Arial",
            italic=True,
            size=9,
            color="555555"
        )

        c2.alignment = LFT


# ─────────────────────────────────────────────────────────────────────────────
# Workbook generation
# ─────────────────────────────────────────────────────────────────────────────

if 'all_results' not in dir() or not all_results:

    print("⚠️ No results. Run Step 3 first.")

else:

    wb = openpyxl.Workbook()

    now_str = datetime.now().strftime("%Y-%m-%d %H:%M")

    # ─────────────────────────────────────────────────────────────────────────
    # Summary Sheet
    # ─────────────────────────────────────────────────────────────────────────

    ws_sum = wb.active
    ws_sum.title = "Summary"

    title_block(
        ws_sum,
        "Chandra OCR — GRN Extraction Summary",
        subtitle=f"Generated: {now_str}"
    )

    summary_headers = [
        "#",
        "File Name",
        "Convert",
        "Extract",
        "Quality",
        "Line Items",
        "Errors"
    ]

    summary_rows = []

    total_items = 0

    for i, rec in enumerate(all_results, 1):

        conv = rec.get("convert_result")
        items = rec.get("line_items", [])

        total_items += len(items)

        summary_rows.append([
            i,
            rec["filename"],
            "OK" if conv else "FAIL",
            "OK" if items else "FAIL",
            conv.get("parse_quality_score", "—") if conv else "—",
            len(items),
            "; ".join(rec.get("errors", []))
        ])

    write_table(ws_sum, summary_headers, summary_rows, start_row=4)

    # ─────────────────────────────────────────────────────────────────────────
    # Combined Items Sheet
    # ─────────────────────────────────────────────────────────────────────────

    ws_all = wb.create_sheet("All Line Items")

    title_block(
        ws_all,
        "All GRN Line Items"
    )

    item_headers = [
        "Source File",
        "Medicine Name",
        "Batch No",
        "Expiry Date",
        "Quantity",
        "Free Quantity",
        "Rate",
        "MRP",
        "GST %",
        "Amount",
        "HSN Code"
    ]

    all_rows = []

    for rec in all_results:

        for item in rec.get("line_items", []):

            all_rows.append([
                rec["filename"],
                item.get("medicine_name"),
                item.get("batch_no"),
                item.get("expiry_date"),
                item.get("quantity"),
                item.get("free_quantity"),
                item.get("rate"),
                item.get("mrp"),
                item.get("gst_percent"),
                item.get("amount"),
                item.get("hsn_code"),
            ])

    if all_rows:
        write_table(ws_all, item_headers, all_rows, start_row=4)
    else:
        ws_all.cell(row=4, column=1).value = "No items extracted."

    # ─────────────────────────────────────────────────────────────────────────
    # Per-file Sheets
    # ─────────────────────────────────────────────────────────────────────────

    for rec in all_results:

        stem = Path(rec["filename"]).stem[:24]

        safe = "".join(
            c if c not in r'\/*?[]:"' else "_"
            for c in stem
        )

        ws = wb.create_sheet(f"{safe[:28]}")

        items = rec.get("line_items", [])

        title_block(
            ws,
            f"{rec['filename']} — Line Items",
            subtitle=f"Rows extracted: {len(items)}"
        )

        rows = []

        for item in items:

            rows.append([
                item.get("medicine_name"),
                item.get("batch_no"),
                item.get("expiry_date"),
                item.get("quantity"),
                item.get("free_quantity"),
                item.get("rate"),
                item.get("mrp"),
                item.get("gst_percent"),
                item.get("amount"),
                item.get("hsn_code"),
            ])

        headers = [
            "Medicine Name",
            "Batch No",
            "Expiry Date",
            "Quantity",
            "Free Quantity",
            "Rate",
            "MRP",
            "GST %",
            "Amount",
            "HSN Code"
        ]

        if rows:
            write_table(ws, headers, rows, start_row=4)
        else:
            ws.cell(row=4, column=1).value = "No items extracted."

    # ─────────────────────────────────────────────────────────────────────────
    # Save Workbook
    # ─────────────────────────────────────────────────────────────────────────

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    output_path = f"chandra_grn_output_{ts}.xlsx"

    wb.save(output_path)

    print(f"✅ Excel workbook saved: {output_path}")
    print(f"📦 Total line items exported: {total_items}")
    print("⬇️ Downloading workbook...")

    colab_files.download(output_path)

✅ Excel workbook saved: chandra_grn_output_20260518_142357.xlsx
📦 Total line items exported: 10
⬇️ Downloading workbook...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 🛠️ Troubleshooting

| Problem | Likely Cause | Fix |
|---------|-------------|-----|
| `CHANDRA_API_KEY not found` | Secret not added | Left sidebar → 🔑 Secrets → add `CHANDRA_API_KEY` |
| `Submit failed [401]` | API key wrong or expired | Check key at https://www.datalab.to/app/keys |
| `Rate limit exceeded (429)` | Too many requests | Wait 30–60 seconds, then re-run Step 3 |
| `Timeout after 180s` | Slow server or very large image | Increase `TIMEOUT_SEC` in Step 3 config |
| `No line items found` | Schema mismatch for your document | Edit `EXTRACT_SCHEMA` in Step 3 config to match your fields |
| `Cannot decode image` | Corrupt or unsupported format | Re-export the image as a clean JPG or PNG |
| Download didn't start | Pop-up blocked in browser | Allow pop-ups for colab.research.google.com |

---

## 💡 Tips for Best Results

- **Resolution:** 150–300 DPI minimum — higher is better for small or handwritten text
- **Lighting:** Avoid shadows, glare, or heavy page curl when photographing documents
- **Mode:** Use `accurate` for invoices; `fast` for quick tests
- **Schema:** The more descriptive your field `description` values, the better the extraction
- **Both modes:** Running both `convert` and `extract` gives the richest output —  
  `convert` catches everything; `extract` gives clean structured data

---

> **API Reference:** https://documentation.datalab.to/docs/welcome/api  
> **Playground:** https://www.datalab.to/playground  
> **API Keys:** https://www.datalab.to/app/keys  
> **Support:** support@datalab.to